# Lens Diff Interactive Notebook
### Builds interactive widget for heamtap for diffing jens, tuned lens, logit Lens

In [ ]:
%pip install plotly ipywidgets

In [ ]:
import sys
import importlib
from pathlib import Path

import torch
import transformers

ROOT = Path("/media/am/AM/mi-lens")

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "lenses" / "jacobian_lens"))
sys.path.insert(0, str(ROOT / "lenses" / "tuned_logit_lens"))

import jlens
from tuned_lens import TunedLens
import plotting.lens_diff_widget

importlib.reload(plotting.lens_diff_widget)
from plotting.lens_diff_widget import build_lens_comparison_widget

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
JLENS_PATH = ROOT / "tmp" / "jlens_out" / "ckpt.pt"
TUNED_LENS_PATH = ROOT / "tmp" / "tuned_lens_tinyllama_wikitext100_v2"

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
).cpu()

model = jlens.from_hf(hf_model, tokenizer)
lens = jlens.JacobianLens.load(str(JLENS_PATH))
tuned_lens = TunedLens.from_model_and_pretrained(hf_model, str(TUNED_LENS_PATH))

In [ ]:
widget = build_lens_comparison_widget(
    model,
    lens,
    tokenizer,
    prompt="Fact: The currency used in the country shaped like a boot is",
    tuned_lens=tuned_lens,
    last_n_positions=12,
    top_k=8,
    correct_k=5,
)

widget